# Kaggle Pilot — `beyond-plausibility` / `faithful-ids` (REAL vertical slice)

**This notebook is only an execution wrapper.** It clones the repository at a pinned tag, installs
it, downloads/points at CICIDS2017, and invokes the repository's own CLI. All data processing,
detector training, SHAP, LLM generation, extraction, verification, metrics, and artifact writing
happen **inside the repository** (`faithfulids.orchestration.execute.run_pilot`).

**What runs (real):** CICIDS2017 -> XGBoost -> exact TreeSHAP -> B0-B4 explanations (B2/B3/B4 via one
4-bit instruct LLM) -> rule-assisted extraction -> Layer-1/Layer-2/cost metrics -> a hash-manifested
run in `runs/EXP-PILOT-001/`, then Friedman+Nemenyi across B0-B4, a CD diagram, a coverage-risk
curve, and a per-generator faithfulness table.

**Kaggle-fit simplifications (documented, firewall intact, NON-CITABLE):** to load only ONE LLM, the
extractor runs rule-assisted (its config already declares `rule_assisted: true`) and B4's verifier is
rule-based grounding against the SHAP evidence. Pilot outputs live in the `pilot` tier / `pilot` seed
section, separate from Tier-A, and must be excluded from confirmatory analysis.


## 1. Environment setup

In [ ]:
# Launcher parameters ONLY (no experiment configuration is defined here).
import os
PINNED_REF = 'main'   # NON-CITABLE quick test: 'main' is a moving branch. Tag a SHA (e.g. pilot-v2.1) for a citable run.

# ---------------- model selection: the ONLY lines to edit per run ----------------
MODEL = 'qwen3_32b_4bit'   # any id in configs/llms/ - the cell after install lists them
N     = '60'                 # explained flows. 150 = standard pilot; 40-60 = smoke
                             # (recommended first run for 30B-class models: estimate
                             # wall-clock before committing to full N)
# ----------------------------------------------------------------------------------

os.environ['FAITHFULIDS_REF']       = PINNED_REF
os.environ['FAITHFULIDS_PILOT_LLM'] = MODEL
os.environ['FAITHFULIDS_PILOT_N']   = N
os.environ['FAITHFULIDS_MAX_ROWS']  = '200000'  # cap rows loaded before sampling (memory)
os.environ['FAITHFULIDS_ENFORCE_COMPETENCE'] = '0'  # REPORT per-family competence, don't HALT (exploratory pilot)
os.environ['FAITHFULIDS_DEVICE_MAP'] = 'auto'   # shard across all visible GPUs; 32B-4bit (~18 GB) NEEDS both T4s.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # reduce fragmentation on tightly-packed GPUs
                                                # 'single' pins GPU 0 (fine for <=8B).
EXP_PILOT = 'EXP-PILOT-001'
print('pin:', PINNED_REF, '| model:', MODEL, '| N:', N,
      '| device_map:', os.environ['FAITHFULIDS_DEVICE_MAP'],
      '| enforce_competence:', os.environ['FAITHFULIDS_ENFORCE_COMPETENCE'])


In [ ]:
%%bash
set -e
rm -rf /kaggle/working/faithful-ids
git clone --quiet https://github.com/SLIMIHamda/faithful-ids.git /kaggle/working/faithful-ids
cd /kaggle/working/faithful-ids && git checkout --quiet "${FAITHFULIDS_REF:-pilot-v2}"
echo 'checked out:' $(git rev-parse HEAD) 'ref='"${FAITHFULIDS_REF:-pilot-v2}"


In [ ]:
%%bash
# Install the repo WITHOUT its exact pins (Kaggle already ships numpy/pandas/sklearn/scipy/xgboost/
# shap/torch/transformers). Add only what Kaggle may lack.
cd /kaggle/working/faithful-ids
pip install -q -e . --no-deps 2>&1 | tail -n 2
pip install -q pyyaml jsonschema import-linter bitsandbytes accelerate 2>&1 | tail -n 2
# PIN transformers < 5 (generator sessions). Kaggle ships transformers 5.0.0, whose
# bnb 4-bit load path regressed (huggingface/transformers#43032) and OOM'd/crashed the
# 4-bit generator load — hit by the Qwen3-32B run, which worked around it by hand. Baking
# the pin in here removes the manual step. `<5` (not the repo's declared 4.46.2, which
# predates Qwen3 support) resolves to a recent, Qwen3-capable 4.x. NOTE: replay-only
# sessions (kaggle_rescore_launcher) load no model and do NOT need this; LLM-assisted
# EXTRACTION (Gemma-4) needs v5 instead and must run in its own session.
pip install -q "transformers<5" 2>&1 | tail -n 2
python -c "import xgboost, shap, torch, transformers; print('xgboost',xgboost.__version__,'| torch',torch.__version__,'| transformers',transformers.__version__,'| cuda',torch.cuda.is_available())"
# Record the ACTUAL Kaggle environment AFTER all pins, so environment.txt matches what
# runs (the 32B export recorded transformers 5.0.0 from a PRE-pin freeze while the run
# actually used 4.57.6 — this ordering fixes that provenance wart).
python environment/env-fingerprint.py > /kaggle/working/env-fingerprint.json
pip freeze > /kaggle/working/environment.txt
git rev-parse HEAD > /kaggle/working/PINNED_COMMIT.txt
echo 'commit:' $(cat /kaggle/working/PINNED_COMMIT.txt) \
     '| transformers:' $(python -c "import transformers;print(transformers.__version__)")

In [ ]:
# Generator LLMs available at this checkout (edit MODEL in the params cell to switch).
# Fails fast on a typo'd MODEL instead of half an hour into the run.
import glob, os, yaml
sel = os.environ['FAITHFULIDS_PILOT_LLM']
print(f"{'id':<26}{'family':<10}{'quant':<7}{'weights (pinned revision)'}")
ids = []
for p in sorted(glob.glob('/kaggle/working/faithful-ids/configs/llms/*.yaml')):
    c = yaml.safe_load(open(p))
    if c.get('provider') != 'local_open_weights':
        continue  # frontier_api etc. need keys; not the pilot path
    ids.append(c['id'])
    w = c.get('weights', {})
    mark = ' <-- selected' if c['id'] == sel else ''
    print(f"{c['id']:<26}{c.get('model_family','?'):<10}{str(c.get('quantisation')):<7}"
          f"{w.get('hf_repo','?')} @ {str(w.get('revision'))[:10]}{mark}")
assert sel in ids, (f"MODEL={sel!r} has no config in configs/llms/ - "
                    f"pick one of {ids} or add a pinned config to the repo first.")


## 2. Configure the experiment & data

Add a **CICIDS2017** dataset to this notebook (Add Input -> search 'CICIDS2017'); the raw MachineLearning
CSVs or an upstream-corrected variant both work. The cell below auto-detects the directory. The
experiment configuration itself lives in the repo (`experiments/pilot/EXP-PILOT-001_vertical_slice.yaml`).

In [ ]:
import glob, os
# Auto-detect a CICIDS2017 CSV directory under /kaggle/input (override CIC_DIR if needed).
CIC_DIR = os.environ.get('CIC_DIR')
if not CIC_DIR:
    cands = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)})
    def has_label(d):
        f = sorted(glob.glob(os.path.join(d, '*.csv')))[:1]
        return f and 'label' in open(f[0]).readline().lower()
    cands = [d for d in cands if has_label(d)] or cands
    CIC_DIR = cands[0] if cands else ''
assert CIC_DIR, 'No CSV dataset found under /kaggle/input — add a CICIDS2017 dataset as Input.'
os.environ['FAITHFULIDS_DATA_DIR'] = CIC_DIR
print('CICIDS2017 dir:', CIC_DIR)
print('csv files:', len(glob.glob(os.path.join(CIC_DIR, '*.csv'))))
# Optional: HF token for gated models (from Kaggle Secrets, never hard-coded).
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN'); print('HF_TOKEN set')
except Exception as e:
    print('No HF secret (fine unless the model is gated):', type(e).__name__)

In [ ]:
%%bash
cd /kaggle/working/faithful-ids
echo '--- EXP-PILOT-001 (config lives in the repo) ---'
cat experiments/pilot/EXP-PILOT-001_vertical_slice.yaml

## 3. Launch the pipeline (all logic runs INSIDE the repo)

In [ ]:
%%bash --no-raise-error
# Reproducibility gates first.
cd /kaggle/working/faithful-ids
echo '== import contracts =='; lint-imports --config pyproject.toml || true
echo '== firewall =='; python tools/firewall_check.py
echo '== validate configs =='; python -m faithfulids.orchestration.validate_configs

In [ ]:
%%bash --no-raise-error
# RUN 1 (3B). Trains XGBoost, computes TreeSHAP, generates B0-B4 (B2-B4 via the
# generator LLM selected by $FAITHFULIDS_PILOT_LLM), extracts claims, computes
# Layer-1 / Layer-2 (eps_att + eps_model, prob AND margin) / cost, writes a
# hash-manifested run. Competence is REPORTED (not enforced) per cell 2.
cd /kaggle/working/faithful-ids
python -m faithfulids.orchestration.cli run --experiment EXP-PILOT-001


## 4. Analysis + interpretable results (produced by the repo)

In [ ]:
%%bash --no-raise-error
cd /kaggle/working/faithful-ids
python -m analysis.run --config pilot_faithfulness
python -m analysis.run --config pilot_coverage
python -m paper.figures.build
python -m paper.tables.build

In [ ]:
# Display the REAL results the repo produced (reading/rendering only — no computation here).
import json, os
from IPython.display import Image, display
REPO = '/kaggle/working/faithful-ids'
res = json.load(open(os.path.join(REPO, 'analysis/outputs/pilot_faithfulness/results.json')))['result']
print('Friedman across B0-B4 on Layer-1 mention F1 (real CICIDS2017 x XGBoost x TreeSHAP):')
print('  statistic =', round(res['statistic'],3), ' p =', round(res['pvalue'],4),
      ' critical difference =', round(res['critical_difference'],3))
for m, r, s in sorted(zip(res['methods'], res['avg_ranks'], res['scores']), key=lambda x: x[1]):
    print(f"  {m:14s} avg_rank={r:5.2f}  mean_F1={sum(s)/len(s):.3f}")
cov = json.load(open(os.path.join(REPO, 'analysis/outputs/pilot_coverage/results.json')))['result']
print('\nB4 coverage-risk AURC =', round(cov['aurc'], 4))
for p in ['paper/figures/generated/fig_pilot_cd.png', 'paper/figures/generated/fig_pilot_coverage.png']:
    fp = os.path.join(REPO, p)
    if os.path.exists(fp): display(Image(fp))

## 5. Validate the artifact chain

In [ ]:
%%bash --no-raise-error
cd /kaggle/working/faithful-ids
echo '== manifest audit =='; python tools/audit_manifests.py || true
latest=$(ls -1 runs/EXP-PILOT-001 | tail -n 1)
echo "== pilot run: $latest =="
echo 'STATUS:' $(cat runs/EXP-PILOT-001/$latest/STATUS)
echo 'artifacts:'; ls -1 runs/EXP-PILOT-001/$latest/artifacts
echo 'manifest models + inputs:'; python - <<PY
import json; m=json.load(open('runs/EXP-PILOT-001/$latest/MANIFEST.json'))
print(' status  :', m['status'])
print(' models  :', [x['identity'] for x in m['models']])
print(' n inputs:', len(m['inputs']), '(hashed CICIDS2017 CSVs)')
print(' seeds   :', m['randomness'])
PY

In [ ]:
%%bash --no-raise-error
# Hash-verify the run through the repo's READ-ONLY results API.
cd /kaggle/working/faithful-ids
latest=$(ls -1 runs/EXP-PILOT-001 | tail -n 1)
python - <<PY
from faithfulids.results import load_run, load_metrics, is_complete_and_verified
rid='$latest'
print('outputs hash-verified:', is_complete_and_verified(rid, runs_root='runs'))
rows=load_metrics(rid, runs_root='runs')
print('metric rows:', len(rows),
      '| layers:', sorted({r['layer'] for r in rows}))
PY

## 6. Export results

In [ ]:
%%bash --no-raise-error
cd /kaggle/working/faithful-ids
rm -rf /kaggle/working/pilot_artifacts && mkdir -p /kaggle/working/pilot_artifacts
cp -r runs /kaggle/working/pilot_artifacts/ 2>/dev/null || true
cp -r analysis/outputs /kaggle/working/pilot_artifacts/analysis_outputs 2>/dev/null || true
cp -r paper/figures/generated paper/tables/generated /kaggle/working/pilot_artifacts/ 2>/dev/null || true
cp /kaggle/working/environment.txt /kaggle/working/env-fingerprint.json /kaggle/working/PINNED_COMMIT.txt /kaggle/working/pilot_artifacts/ 2>/dev/null || true
cd /kaggle/working && zip -qr pilot_artifacts.zip pilot_artifacts && ls -la pilot_artifacts.zip
echo 'Save the notebook output as a Kaggle Dataset, or push runs/ to the DVC remote (REPRODUCING.md).'

## Success criteria

| Criterion | How this pilot validates it |
|---|---|
| Repository executes end-to-end without notebook logic | `run_pilot` runs the whole cell inside the repo |
| RQ0 instrumentation works | corruption operators + meta metrics exercised (`tests/`); extractor runs on real text |
| Extraction quality | rule-assisted extractor parses real B0-B4 explanations into claims (Layer-1) |
| Erasure operator functions | Layer-2 conditional-expectation imputation fit on the real train split |
| Verification pipeline executes | B4 rule-verifier + abstention -> B1 fallback (coverage-risk) |
| Cost per cell measurable | LLM ledger + cost accounting (tokens, coverage) in the run |
| Every artifact reproducible from the manifest | manifest-audit + results-API hash verification; inputs are hashed CSVs |

**Use the pilot only to estimate** effect sizes, variance, required sample sizes, LLM-call budget,
and cost — **not** to decide whether to continue. Keep pilot outputs (tier `pilot`, seeds `pilot`)
separate from Tier-A; exclude them from confirmatory analysis. Model precision is set in the LLM
config's `quantisation` (4-bit here), never inferred. Runs are resumable one cell at a time; `runs/`
is write-once (each invocation mints a new run id).

## 7. Optional run 2 — a second generator in the same session

Set `MODEL_2` below and run to produce another `runs/EXP-PILOT-001/` run in a separate
process (one LLM in GPU memory at a time), e.g. the 3B end of a within-family scale
bracket. Repeat the cell with different values for more runs; every run id is new
(`runs/` is write-once) and the export cell above bundles them all.


In [ ]:
%%bash --no-raise-error
MODEL_2=qwen2_5_3b_instruct   # <- the only line to edit for run 2
cd /kaggle/working/faithful-ids
test -f "configs/llms/${MODEL_2}.yaml" || { echo "no config for ${MODEL_2}"; exit 1; }
export FAITHFULIDS_PILOT_LLM="$MODEL_2"
python -m faithfulids.orchestration.cli run --experiment EXP-PILOT-001
echo '--- runs present ---'; ls -1 runs/EXP-PILOT-001
